In [ ]:
import sys
import os
sys.path.append("..")
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Load data Temporal
data_original = torch.load("../wandb_logs/knn-mlp-regression-relative-multiseed/evaluations/6n50ipg0/predictions.pt")
data_both = torch.load("../wandb_logs/knn-mlp-regression-relative-multiseed/evaluations/7gp7i7wa/predictions.pt")

data = {
    "original": data_original,
    "both_augmented": data_both,
}

for split in ["train", "val"]:
    n_dims = data_original[split]["Y"].shape[1]
    fig, axes = plt.subplots(1, n_dims, figsize=(6*n_dims, 5))
    axes = np.atleast_1d(axes)

    for d, ax in enumerate(axes):
        mse_text = []  # store MSEs for legend or annotation
        for key, val in data.items():
            Y = val[split]["Y"].cpu().numpy()
            preds = val[split]["preds"].cpu().numpy()
            mse = np.mean((Y[:, d] - preds[:, d])**2)
            # mse_text.append(f"{key}: {mse:.4f}")
            ax.scatter(Y[:, d], preds[:, d], alpha=0.4, label=f"{key} (MSE={mse:.4f})")

        # Diagonal reference line
        min_val, max_val = Y[:, d].min(), Y[:, d].max()
        ax.plot([min_val, max_val], [min_val, max_val], "r--")
        # ax.axis("equal")

        # Titles and labels
        ax.set_title(f"{split.upper()} dim {d}")
        ax.set_xlabel("True")
        ax.set_ylabel("Predicted")
        ax.legend()


    plt.tight_layout()
    plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Load data
data_original = torch.load("../wandb_logs/knn-mlp-regression-relative-multiseed/evaluations/6n50ipg0/predictions.pt")
data_both = torch.load("../wandb_logs/knn-mlp-regression-relative-multiseed/evaluations/7gp7i7wa/predictions.pt")

data = {
    "original": data_original,
    "both_augmented": data_both,
}


# --- Error histograms ---
for split in ["train", "val"]:
    n_dims = data_original[split]["Y"].shape[1]
    fig, axes = plt.subplots(1, n_dims, figsize=(6*n_dims, 4))
    axes = np.atleast_1d(axes)

    for d, ax in enumerate(axes):
        for key, val in data.items():
            Y = val[split]["Y"].cpu().numpy()
            preds = val[split]["preds"].cpu().numpy()
            errors = preds[:, d] - Y[:, d]
            ax.hist(errors, bins=40, alpha=0.5, label=key, density=True)
        
        ax.set_title(f"{split.upper()} dim {d} — Error Distribution")
        ax.set_xlabel("Prediction Error (pred - true)")
        ax.set_ylabel("Density")
        ax.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
for split in ["train", "val"]:
    n_dims = data_original[split]["Y"].shape[1]
    fig, axes = plt.subplots(n_dims, 1, figsize=(10, 5*n_dims))
    axes = np.atleast_1d(axes)

    for d, ax in enumerate(axes):

        ax.plot(data_original[split]["data_array"]['t'][:],data_original[split]["preds"][:,0], label="original")
        ax.plot(data_original[split]["data_array"]['t'][:],data_both[split]["preds"][:,0], label="both_augmented")
        ax.plot(data_original[split]["data_array"]['t'][:],data_original[split]["Y"][:,0], color='black',label="GT")
            
        
        ax.set_title(f"{split.upper()} dim {d} — predictions over time")
        ax.set_xlabel("Time")
        ax.set_ylabel("Prediction")
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Load your data
data_original = torch.load("../wandb_logs/knn-mlp-regression-relative-multiseed/evaluations/6n50ipg0/predictions.pt")
data_both = torch.load("../wandb_logs/knn-mlp-regression-relative-multiseed/evaluations/7gp7i7wa/predictions.pt")

# Choose split and dimension
split = "val"
dim = 0

# Extract data
t = data_original[split]["data_array"]["t"]
Y = data_original[split]["Y"].cpu().numpy()[:, dim]
preds_original = data_original[split]["preds"].cpu().numpy()[:, dim]
preds_both = data_both[split]["preds"].cpu().numpy()[:, dim]

# Animation parameters
window_size = 1000
step_size = 200           # how far to move the window each frame
t_max = 20000            # total duration (samples)
fps = 3

# Create figure
fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlim(t[0], t[window_size])
y_min = min(Y.min(), preds_original.min(), preds_both.min())
y_max = max(Y.max(), preds_original.max(), preds_both.max())
ax.set_ylim(y_min, y_max)

gt_line, = ax.plot([], [], color='black', label="GT")
orig_line, = ax.plot([], [], label="original")
both_line, = ax.plot([], [], label="both_augmented")
ax.legend()
ax.set_xlabel("Time")
ax.set_ylabel(f"Output dim {dim}")
ax.set_title("Prediction Over Time (Sliding Window)")

# Update function for animation
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(t):  # stop when we reach the end
        end = len(t)
    gt_line.set_data(t[start:end], Y[start:end])
    orig_line.set_data(t[start:end], preds_original[start:end])
    both_line.set_data(t[start:end], preds_both[start:end])
    ax.set_xlim(t[start], t[end-1])
    return gt_line, orig_line, both_line

# Build animation
# frames = (t_max - window_size) // step_size
# ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

# plt.show()
# ani.save("time_series_animation.gif", writer="pillow", fps=fps)

# HTML(ani.to_jshtml())

In [ ]:
def compute_vector_errors(preds, Y):
    """
    Compute multiple vector prediction metrics.

    Args:
        preds: [N, D] predicted vectors
        Y: [N, D] ground truth vectors
    Returns:
        dict with metrics
    """
    import numpy as np

    errors = np.linalg.norm(preds - Y, axis=1)
    L1 = np.mean(np.abs(preds - Y))
    L1_GT = np.mean(np.abs(np.mean(Y, axis=0) - Y))
    L2 = np.mean((preds - Y)**2)
    L2_GT = np.mean((np.mean(Y, axis=0) - Y)**2)
    EPE = np.mean(errors)
    EPE_GT = np.mean(np.linalg.norm(np.mean(Y, axis=0) - Y, axis=1))
    acc_1px = np.mean(errors < 1.0)
    acc_2px = np.mean(errors < 2.0)
    acc_3px = np.mean(errors < 3.0)

    dot = np.sum(preds * Y, axis=1)
    norm_pred = np.linalg.norm(preds, axis=1)
    norm_true = np.linalg.norm(Y, axis=1)
    eps = 1e-8
    cosine = dot / (norm_pred * norm_true + eps)
    cosine = np.clip(cosine, -1.0, 1.0)
    angular_error = np.degrees(np.arccos(cosine))
    mean_angle = np.mean(angular_error)
    
    Y_mean = np.mean(Y, axis=0, keepdims=True).repeat(Y.shape[0], axis=0)
    dot = np.sum(Y_mean * Y, axis=1)
    norm_Y_mean = np.linalg.norm(Y_mean, axis=1)
    norm_true = np.linalg.norm(Y, axis=1)
    eps = 1e-8
    cosine = dot / (norm_Y_mean * norm_true + eps)
    cosine = np.clip(cosine, -1.0, 1.0)
    angular_error = np.degrees(np.arccos(cosine))
    mean_angle_GT = np.mean(angular_error)
    

    return {
        "L1": L1,
        "L1_GT": L1_GT,
        "L2": L2,
        "L2_GT": L2_GT,
        "EPE": EPE,
        "EPE_GT": EPE_GT,
        "1px_acc": acc_1px,
        "2px_acc": acc_2px,
        "3px_acc": acc_3px,
        "angular_error_deg": mean_angle,
        "angular_error_deg_GT": mean_angle_GT
    }
    
for key, val in data.items():   
    Y = val["val"]["Y"].cpu().numpy()
    preds = val["val"]["preds"].cpu().numpy()
    metrics = compute_vector_errors(preds, Y)
    print(f"Metrics for {key}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")


In [ ]:
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Choose split and model
split = "val"
model_key = "both_augmented"  # or "original"

# Load arrays
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x = data[model_key][split]["data_array"]["x"]
y = data[model_key][split]["data_array"]["y"]

# --- Compute endpoint error (Euclidean distance)
errors = np.linalg.norm(preds - Y, axis=1)

# Animation parameters
window_size = 1000
step_size = 200        # controls how much we move per frame
fps = 10

# --- Create figure
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter([], [], c=[], cmap='plasma', vmin=0, vmax=np.percentile(errors, 95), s=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Spatial Error Animation ({model_key}, {split})")

# Precompute limits
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Endpoint Error (EPE)")

# --- Update function
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(errors):
        end = len(errors)
    sc.set_offsets(np.c_[x[start:end], y[start:end]])
    sc.set_array(errors[start:end])
    ax.set_title(f"Samples {start}-{end} | {model_key} ({split})")
    return sc,

# Number of frames
frames = (len(errors) - window_size) // step_size

ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

plt.show()
ani.save(f"error_animation_{model_key}_{split}.gif", writer="pillow", fps=fps)

HTML(ani.to_jshtml())

In [ ]:
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Choose split and model
split = "val"
model_key = "original"

# Load arrays
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x = data[model_key][split]["data_array"]["x"]
y = data[model_key][split]["data_array"]["y"]

# --- Compute endpoint error (Euclidean distance)
errors = np.linalg.norm(preds - Y, axis=1)

# Animation parameters
window_size = 1000
step_size = 200        # controls how much we move per frame
fps = 10

# --- Create figure
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter([], [], c=[], cmap='plasma', vmin=0, vmax=np.percentile(errors, 95), s=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Spatial Error Animation ({model_key}, {split})")

# Precompute limits
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Endpoint Error (EPE)")

# --- Update function
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(errors):
        end = len(errors)
    sc.set_offsets(np.c_[x[start:end], y[start:end]])
    sc.set_array(errors[start:end])
    ax.set_title(f"Samples {start}-{end} | {model_key} ({split})")
    return sc,

# Number of frames
frames = (len(errors) - window_size) // step_size

ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

plt.show()
ani.save(f"error_animation_{model_key}_{split}.gif", writer="pillow", fps=fps)

HTML(ani.to_jshtml())

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Choose split and model
split = "val"
model_key = "both_augmented"  # or "original"

# Load arrays
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x = data[model_key][split]["data_array"]["x"]
y = data[model_key][split]["data_array"]["y"]

# --- Compute angular error (in degrees)
dot = np.sum(preds * Y, axis=1)
norm_pred = np.linalg.norm(preds, axis=1)
norm_true = np.linalg.norm(Y, axis=1)
eps = 1e-8
cosine = dot / (norm_pred * norm_true + eps)
cosine = np.clip(cosine, -1.0, 1.0)
angular_error = np.degrees(np.arccos(cosine))

# Animation parameters
window_size = 1000
step_size = 200
fps = 10

# --- Create figure
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter([], [], c=[], cmap='turbo', vmin=0, vmax=np.percentile(angular_error, 95), s=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Angular Error Animation ({model_key}, {split})")

# Precompute limits
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Angular Error (°)")

# --- Update function
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(angular_error):
        end = len(angular_error)
    sc.set_offsets(np.c_[x[start:end], y[start:end]])
    sc.set_array(angular_error[start:end])
    ax.set_title(f"Samples {start}-{end} | {model_key} ({split})")
    return sc,

# Number of frames
frames = (len(angular_error) - window_size) // step_size

ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

plt.show()
ani.save(f"angular_error_animation_{model_key}_{split}.gif", writer="pillow", fps=fps)
HTML(ani.to_jshtml())

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Choose split and model
split = "val"
model_key = "original"

# Load arrays
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x = data[model_key][split]["data_array"]["x"]
y = data[model_key][split]["data_array"]["y"]

# --- Compute angular error (in degrees)
dot = np.sum(preds * Y, axis=1)
norm_pred = np.linalg.norm(preds, axis=1)
norm_true = np.linalg.norm(Y, axis=1)
eps = 1e-8
cosine = dot / (norm_pred * norm_true + eps)
cosine = np.clip(cosine, -1.0, 1.0)
angular_error = np.degrees(np.arccos(cosine))

# Animation parameters
window_size = 1000
step_size = 200
fps = 10

# --- Create figure
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter([], [], c=[], cmap='turbo', vmin=0, vmax=np.percentile(angular_error, 95), s=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Angular Error Animation ({model_key}, {split})")

# Precompute limits
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Angular Error (°)")

# --- Update function
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(angular_error):
        end = len(angular_error)
    sc.set_offsets(np.c_[x[start:end], y[start:end]])
    sc.set_array(angular_error[start:end])
    ax.set_title(f"Samples {start}-{end} | {model_key} ({split})")
    return sc,

# Number of frames
frames = (len(angular_error) - window_size) // step_size

ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

plt.show()
ani.save(f"angular_error_animation_{model_key}_{split}.gif", writer="pillow", fps=fps)
HTML(ani.to_jshtml())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_quiver(X, Y, title):
    plt.figure(figsize=(6,6))
    plt.quiver(X[:,0], X[:,1], Y[:,0], Y[:,1], angles="xy", scale_units="xy", scale=1, alpha=0.5)
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.axis("equal")
    plt.show()

# Example on test
X = data["val"]["X"].numpy()[:, :2]   # assuming first two cols are coordinates
Y_true = data["val"]["Y"].numpy()
Y_pred = data["val"]["preds"].numpy()

idx_range = np.arange(2000,2100)
plot_quiver(X[idx_range,:], Y_true[idx_range,:], "Ground Truth velocity")
plot_quiver(X[idx_range,:], Y_pred[idx_range,:], "Predicted velocity")
